# Libraries Installation

In [ ]:
!pip install langchain langchain-community transformers torch sentence-transformers faiss-cpu pypdf accelerate
!pip install -U langchain-text-splitters
!pip install -U sentence-transformers langchain
!pip install langchain_core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.2/332.2 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.16
    Uninstalling langchain-core-1.2.16:
      Successfully uninstalled langchain-core-1.2.16
ERROR: pip's dependency resolver does not currently take into account all the packages that are ins

In [ ]:
!pip install fastapi uvicorn openai pydantic

In [ ]:
!pip install flask pyngrok

In [ ]:
!ngrok config add-authtoken 39UCYux7XvYgWDJqR7dBvhSGpK7_47XEnz8ujfxK5YXcX3hnq

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


# Load the llm

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline

model_id = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    #device_map="auto",
    torch_dtype="auto"
).to("cuda")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    temperature=0.2
)

llm = HuggingFacePipeline(pipeline=pipe)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
/tmp/ipykernel_240/631689166.py:21: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


# Chat

In [ ]:
from pydantic import BaseModel
from typing import List

class Review(BaseModel):
    ReviewId: int
    JobId: int
    ProviderId: int
    CustomerId: int
    Rating: int
    Comment: str
    CreatedAt: str

class ReviewSummaryRequest(BaseModel):
    reviews: List[Review]

class ProviderSummary(BaseModel):
    providerId: str
    summary: str

class ReviewSummaryResponse(BaseModel):
    summaries: List[ProviderSummary]


In [ ]:
from collections import defaultdict
from transformers import pipeline

def summarize_reviews(reviews):
    joined=""
    for r in reviews:
        joined=joined+"\n"+r["Comment"]
    prompt = f"""
    You are an assistant that summarizes customer reviews.

    Task:
    Given multiple customer comments about the same service provider, produce ONE brief summary (2-3 sentences).

    Rules:
    - Write a single short paragraph (2–4 sentences).
    - Capture the overall sentiment (positive, negative, or mixed).
    - Mention common strengths and common complaints if they exist.
    - Do NOT quote individual reviews.
    - Do NOT invent information.
    - Be neutral and factual.
    - Do NOT use bullet points.

    Reviews:
    {joined}

    Summary:
    """
    #summarizer = pipeline("text-generation", model=llm)
    summary=llm.invoke(prompt)
    return summary

In [ ]:
from pydantic import BaseModel
from typing import Optional, List, Dict, Any
class ChatRequest(BaseModel):
    user_id: int
    role: str   # "customer" or "provider"
    message: str
    context: Optional[Dict[str, Any]] = None


class ToolResult(BaseModel):
    tool_name: str
    #result: Dict[str, Any]
    result: list
    question: str

In [ ]:
Customer_Prompt = """
You are an AI assistant for a service marketplace platform.

You MUST respond in valid JSON format only.

If the Customer asks to get providers base on location:
{
  "type": "tool_call",
  "tool_name": "get_providers_by_location",
  "arguments": { "location": <location> }
}

If the Customer asks about top rated providers:
{
  "type": "tool_call",
  "tool_name": "get_top_rated_providers",
  "arguments": { "limit": <limit=5> }
}

If the Customer asks about the service categories provided in the marketplace:
{
  "type": "tool_call",
  "tool_name": "get_all_categories",
}

If the Customer asks ANYTHING outside the three tool categories above, you MUST NOT answer.
You MUST ONLY return:
{
  "type": "final_answer",
  "content":"this question can not be answered"
}
Do not provide any other text, explanation, or information.
Do not attempt to answer factual questions, even if you know the answer.
"""

Provider_Prompt = """
You are an AI assistant for a service marketplace platform.

You MUST respond in valid JSON format only.

If the Service Provider asks about top rated providers:
{
  "type": "tool_call",
  "tool_name": "get_top_rated_providers",
  "arguments": { "limit": <limit=5> }
}

If the Service Provider asks about the service categories provided in the marketplace:
{
  "type": "tool_call",
  "tool_name": "get_all_categories",
}

If the Service Provider asks about his Summary provided in the marketplace:
{
  "type": "tool_call",
  "tool_name": "get_my_summary",
}

If the Service Provider asks ANYTHING outside the three tool categories above, you MUST NOT answer.
You MUST ONLY return:
{
  "type": "final_answer",
  "content":"this question can not be answered"
}
Do not provide any other text, explanation, or information.
Do not attempt to answer factual questions, even if you know the answer.
"""

SYSTEM_PROMPT = """
    You are formatting the final response to the user.
    You will be given the user question with some data to give a good answer
    """

In [ ]:
def chat(request: ChatRequest):
    if request.role == "Customer":
      prompt = f"""
{Customer_Prompt}

User Role: {request.role}
User ID: {request.user_id}

User Question:
{request.message}

Your Answer:
"""

    else:
      prompt=f"""
{Provider_Prompt}

User Role: {request.role}
User ID: {request.user_id}

User Question:
{request.message}

Your Answer:
"""
    raw_output = llm.invoke(prompt)
    return raw_output

def final_answer(request: ToolResult):
    prompt = f"""
{SYSTEM_PROMPT}

Tool used to help a good answer: {request.tool_name}
Result of the tool: {request.result}

User Question:
{request.question}

Your Answer:
"""
    raw_output = llm.invoke(prompt)
    return raw_output

In [ ]:
from flask import Flask, request, jsonify
from pyngrok import ngrok

app = Flask(__name__)



@app.route('/chat', methods=['POST'])
def chat_fc():
    chatRequest=request.get_json(force=True)
    chatRequest=ChatRequest(**chatRequest)
    res=chat(chatRequest)

    text=res
    if "Your Answer:" in text:
      response = text.split("Your Answer:")[-1].strip()
      print("Response: ",response)

    #print("Summary:",summary)

    return response
    #return jsonify({response})

@app.route('/chat/tool-response', methods=['POST'])
def final_answer_fc():
  chat_request = request.get_json(force=True)
  print("Request: ",chat_request)
  chat_request = ToolResult(**chat_request)
  res=final_answer(chat_request)

  text=res
  if "Your Answer:" in text:
      response = text.split("Your Answer:")[-1].strip()
      print("Response: ",response)
  return jsonify({"content":response})


@app.route('/summarize', methods=['POST'])
def summarize():
    reviews=request.get_json(force=True)
    print("Reviews: ",reviews)
    summary = summarize_reviews(reviews)

    text=summary
    # Extract only the part after "Summary:"
    if "Summary:" in text:
      response = text.split("Summary:")[-1].strip()

    print("Summary:",summary)

    return jsonify({"summary": response})


# Open a tunnel on port 5000
public_url = ngrok.connect(5000)
print("Public URL:", public_url)

app.run(port=5000)

Public URL: NgrokTunnel: "https://unobstruently-leafiest-mary.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
INFO:werkzeug:127.0.0.1 - - [10/Mar/2026 14:40:36] "POST /chat HTTP/1.1" 200 -


Response:  {
  "type": "tool_call",
  "tool_name": "get_providers_by_location",
  "arguments": { "location": "Tripoli" }
}


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Request:  {'tool_name': 'get_providers_by_location', 'result': [{'providerId': 1, 'userId': 4, 'biography': '....', 'yearOfExperience': 5, 'location': 'Tripoli', 'trustScore': None, 'providerServices': None, 'reviews': None, 'reviewsSummaries': None, 'jobAssignments': None}, {'providerId': 2, 'userId': 5, 'biography': '....', 'yearOfExperience': 10, 'location': 'Tripoli', 'trustScore': None, 'providerServices': None, 'reviews': None, 'reviewsSummaries': None, 'jobAssignments': None}], 'question': 'could you give me the providers that live in Tripoli'}


...

Response:  There are 3 service categories provided in this marketplace: Plumber, Electrical, and Mechanics.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
INFO:werkzeug:127.0.0.1 - - [10/Mar/2026 14:44:54] "POST /chat HTTP/1.1" 200 -


Response:  {
  "type": "final_answer",
  "content":"you're welcome"
}


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
INFO:werkzeug:127.0.0.1 - - [10/Mar/2026 14:45:15] "POST /chat HTTP/1.1" 200 -


Response:  {
  "type": "final_answer",
  "content":"this question can not be answered"
}
